# FBBench zero-shot: Gemini reference (CPU)

**Generated from `kernels/fbbench-zeroshot`** by changing only the config cell, so
Gemini sees the same questions, frames (cap, resize) and prompt as the local models.
Edit the zero-shot notebook, not this one.

Gemini is a reference point, not a backbone: it cannot be fine-tuned or inspected
for the faithfulness study. Answers are constrained to the option letters by an
enum response schema; tokens come from the API's usage metadata.

**Needs the `GEMINI_API_KEY` secret** (Add-ons → Secrets). Kaggle passes secrets
only to runs started in the editor (**Save Version → Save & Run All**), not to runs
started by `kaggle kernels push`.

**Accelerator:** none. No GPU quota is used.

## 1 · Config

In [ ]:
GH_USER = "gourilaxmi"
GH_REPO = "badminton-agentic-vlm"
BRANCH  = "main"             # results are committed to main

EXPERIMENT  = "e04_gemini"   # closed-model reference on the same 500 questions as e01 / e03
# Backbone screening (Day 4): every model sees the SAME questions, frames and prompt.
# One session runs them in order, so a single queue wait covers the whole sweep.
MODELS      = ["gemini-2.5-pro"]   # the paper's Gemini (38.6% with THEIR sampling); ours is uniform 2 FPS
N_QUESTIONS = 500            # spread round-robin over the 12 sub-tasks; None = all
MAX_MEMORY  = {0: "10GiB", 1: "11GiB"}  # fits 8B fp16 weights; efficient attention needs little headroom
MAX_FRAMES  = 25             # configs/default.yaml budget_frames; 64 OOMs a T4 on long rallies
SHORT_SIDES = [None]         # None = the cache's 448
PUSH_RESULTS = True          # needs GH_TOKEN; ignored without it

TRANSFORMERS = "transformers==5.19.0"   # pinned: the version HFBackend is tested on

## 2 · Code: clone from GitHub, or fall back to the attached snapshot

In [ ]:
import os, sys, json, subprocess, pathlib, datetime

WORK = pathlib.Path("/kaggle/working")
INPUT = pathlib.Path("/kaggle/input")

GH_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    GH_TOKEN = UserSecretsClient().get_secret("GH_TOKEN")
except Exception as e:                                   # noqa: BLE001
    print("no GH_TOKEN secret attached:", type(e).__name__)

API_ONLY = all(m.startswith("gemini-") for m in MODELS)   # no GPU, no weights
GEMINI_API_KEY = None
if any(m.startswith("gemini-") for m in MODELS):
    try:
        GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
    except Exception as e:                               # noqa: BLE001
        raise SystemExit(f"Gemini needs the GEMINI_API_KEY secret ({type(e).__name__}). Secrets only "
                         "reach runs started in the editor: Save Version -> Save & Run All")


def sh(cmd, cwd=None, check=True):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = r.stdout + r.stderr
    if GH_TOKEN:
        out = out.replace(GH_TOKEN, "***")              # never let the token reach the output
    print(out[-3000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd[:3]}")
    return r


if GH_TOKEN:
    REPO = WORK / GH_REPO
    url = f"https://{GH_TOKEN}@github.com/{GH_USER}/{GH_REPO}.git"
    if not REPO.exists():
        sh(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)])
    sh(["git", "config", "user.email", "btp-kaggle-runner@users.noreply.github.com"], cwd=REPO)
    sh(["git", "config", "user.name", "BTP Kaggle Runner"], cwd=REPO)
    COMMIT = sh(["git", "rev-parse", "--short", "HEAD"], cwd=REPO).stdout.strip()
else:
    snap = [p.parents[2] for p in INPUT.rglob("src/bavlm/__init__.py")]  # <snapshot>/src/bavlm/__init__.py
    if not snap:
        raise SystemExit("no GH_TOKEN and no bavlm-src snapshot attached -- see the top cell")
    REPO = snap[0]
    COMMIT = (REPO / "COMMIT").read_text().strip() if (REPO / "COMMIT").exists() else "snapshot"
    PUSH_RESULTS = False

sys.path.insert(0, str(REPO / "src"))
print("code:", REPO, "@", COMMIT)

## 3 · Environment

In [ ]:
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # before CUDA init
sh([sys.executable, "-m", "pip", "install", "-q", TRANSFORMERS, "accelerate"]
   + (["google-genai"] if GEMINI_API_KEY else []))

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
if API_ONLY:
    DTYPE = None
    print("API models only: no GPU needed")
else:
    assert torch.cuda.is_available(), "no GPU -- set the accelerator to T4 x2"
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"cuda:{i}  {p.name}  {p.total_memory / 1e9:.1f} GB  sm_{p.major}{p.minor}")
    DTYPE = "bfloat16" if torch.cuda.get_device_properties(0).major >= 8 else "float16"
    print("dtype:", DTYPE)

## 4 · Data: find the frame cache and the question files

In [ ]:
import time, zipfile

manifests = sorted(INPUT.rglob("frames/manifest.json"))
if not manifests:
    # Kaggle packs a kernel output this large (100k+ files) into one _output_.zip and
    # mounts it unextracted. Unpack once to /tmp so the frames stay out of our own output.
    zips = [z for z in INPUT.rglob("_output_.zip")
            if "frames/manifest.json" in zipfile.ZipFile(z).namelist()]
    assert zips, "frame cache not attached -- add the finebadminton-frame-cache kernel as input"
    CACHE = pathlib.Path("/tmp/frame-cache")
    if not (CACHE / "frames" / "manifest.json").exists():
        t = time.time()
        zipfile.ZipFile(zips[0]).extractall(CACHE)
        print(f"unpacked {zips[0].stat().st_size / 1e9:.1f} GB in {time.time() - t:.0f}s")
    manifests = [CACHE / "frames" / "manifest.json"]
FRAMES = manifests[0].parent
FBBENCH = FRAMES.parent / "fbbench"
cache = json.loads(manifests[0].read_text())
print(f"frames:  {FRAMES}  ({cache['clips']} clips, {cache['frames']} frames, "
      f"{cache['fps']} fps, short side {cache['size']}, revision {cache['revision'][:8]})")
print(f"fbbench: {FBBENCH}")

from bavlm.eval.fbbench import DOMAINS, extract_choice, load_questions, score
from bavlm.models.backend import build_backend

allq = load_questions(FBBENCH)
have = {d.name for d in FRAMES.iterdir() if d.is_dir()}
cached = [q for q in allq if q.clip_id in have]
print(f"{len(allq)} questions, {len(cached)} with cached frames")

## 5 · Mock check: must pass before any GPU time

In [ ]:
key = {q.qid: q.answer for q in cached}
for mode, ok in (("oracle", lambda a: a == 1.0), ("random", lambda a: 0.20 < a < 0.30),
                 ("refuse", lambda a: a == 0.0)):
    be = build_backend({"mock": mode, "seed": 7}, answer_key=key)
    acc = sum(extract_choice(be.generate([], q.prompt, qid=q.qid)) == q.answer for q in cached) / len(cached)
    print(f"{mode:7} {acc:6.1%}")
    if not ok(acc):
        raise SystemExit(f"mock {mode} = {acc:.1%}: the scorer is broken -- stop")

## 6 · The run: resumable, one JSONL row per question

In [ ]:
import gc, itertools, shutil, time

def round_robin(qs, n):
    """Spread a small run over every sub-task instead of taking the first file.
    Deterministic, so every model in the sweep gets exactly the same questions."""
    by_sub = {}
    for q in qs:
        by_sub.setdefault(q.subtask, []).append(q)
    order = [q for grp in itertools.zip_longest(*by_sub.values()) for q in grp if q]
    return order[:n] if n else order


def run_model(model_id):
    """Score every selected question with one model at each short side in SHORT_SIDES.
    Resumable, one JSONL per (model, side); the weights load once for the whole sweep."""
    backend = build_backend({"hf_id": model_id, "dtype": DTYPE, "max_frames": MAX_FRAMES,
                             "short_side": None, "fps": cache["fps"], "max_memory": MAX_MEMORY,
                             "api_key": GEMINI_API_KEY})
    for side in SHORT_SIDES:
        label = model_id + (f"@{side}" if side else "")
        results = WORK / "results" / f"{EXPERIMENT}_{label.split('/')[-1]}.jsonl"
        results.parent.mkdir(parents=True, exist_ok=True)
        done = set()
        if results.exists():
            done = {json.loads(l)["qid"] for l in results.open() if l.strip()}
            print(f"resuming: {len(done)} already done")
        todo = [q for q in round_robin(cached, N_QUESTIONS) if q.qid not in done]
        print(f"== {label}: {len(todo)} to run")
        backend.short_side = side                            # downscale at load, never upscale

        t0 = time.time()
        with results.open("a") as fh:
            for i, q in enumerate(todo, 1):
                t = time.time()
                letters = "".join(chr(ord("A") + k) for k in range(q.n_options))
                err = None
                try:
                    pred = backend.generate(q.load_frames(FRAMES), q.prompt, choices=letters)
                    err = backend.last_stats.get("error")      # e.g. "nonfinite" (fp16 overflow)
                except torch.OutOfMemoryError:
                    # one outlier must not end the run: log it, score it wrong, report the count
                    pred, err = None, "oom"
                    torch.cuda.empty_cache()
                s = backend.last_stats
                fh.write(json.dumps({
                    "qid": q.qid, "clip_id": q.clip_id, "domain": q.domain, "subtask": q.subtask,
                    "gold": q.answer, "pred": pred, "seconds": round(time.time() - t, 3),
                    "n_frames": s["n_frames"], "input_tokens": s["input_tokens"],
                    "visual_tokens": s["visual_tokens"], "fps": s.get("fps"),
                    "frame_wh": s.get("frame_wh"),
                    "choice_logprobs": None if err else s.get("choice_logprobs"), "error": err,
                }) + "\n")
                fh.flush()                                   # survive a kill at any moment
                if i <= 3 or i % 25 == 0 or err:
                    print(f"{i}/{len(todo)}  {q.qid}  gold {q.answer} pred {pred}  "
                          f"{s['n_frames']} frames {s['input_tokens']} tok  {time.time() - t:.1f}s"
                          + (f"  ERROR {err}" if err else ""))
        print(f"done in {(time.time() - t0) / 60:.1f} min")
        RESULTS[label], RUN_SIDE[label] = results, side

    # free both GPUs and the disk before the next model
    del backend
    gc.collect()
    torch.cuda.empty_cache()
    shutil.rmtree(pathlib.Path.home() / ".cache/huggingface/hub" / f"models--{model_id.replace('/', '--')}",
                  ignore_errors=True)


RESULTS, RUN_SIDE = {}, {}
for model_id in MODELS:
    run_model(model_id)

## 7 · Score: per domain always (rule 5), tokens and frames as the cost (rule 6)

In [ ]:
import pandas as pd

agg = dict(n=("correct", "size"), accuracy=("correct", "mean"), frames=("n_frames", "mean"),
           input_tokens=("input_tokens", "mean"), sec_per_q=("seconds", "mean"))
frames_by_model, screen = {}, []
for model_id, path in RESULTS.items():
    df = pd.read_json(path, lines=True)
    df["correct"] = df["pred"] == df["gold"]
    frames_by_model[model_id] = df
    by_domain = df.groupby("domain").agg(**agg).reindex([d for d in DOMAINS if d in set(df.domain)]).round(3)
    by_subtask = df.groupby(["domain", "subtask"]).agg(**agg).round(3)
    by_domain.to_csv(WORK / "results" / f"{EXPERIMENT}_{model_id.split('/')[-1]}_by_domain.csv")
    by_subtask.to_csv(WORK / "results" / f"{EXPERIMENT}_{model_id.split('/')[-1]}_by_subtask.csv")
    s = score(df.to_dict("records"))
    errors = df["error"].value_counts().to_dict() if "error" in df else {}
    print(f"\n===== {model_id}")
    print(by_domain.to_string(), "\n")
    print(by_subtask.to_string(), "\n")
    print(f"overall {s['overall']:.1%} (n={s['n']})  -- never report this alone")
    print("errors:", errors)
    print(f"mean frames {df.n_frames.mean():.1f}  mean input tokens {df.input_tokens.mean():.0f}")
    print("pred distribution:", df.pred.value_counts().to_dict(), " gold:", df.gold.value_counts().to_dict())
    screen.append({"model": model_id.split("/")[-1], **{d: by_domain.accuracy.get(d) for d in DOMAINS},
                   "always_A": round((df.gold == "A").mean(), 3), "errors": sum(errors.values()),
                   "frames": round(df.n_frames.mean(), 1), "input_tokens": round(df.input_tokens.mean()),
                   "sec_per_q": round(df.seconds.mean(), 2)})

screen = pd.DataFrame(screen).set_index("model")
screen.to_csv(WORK / "results" / f"{EXPERIMENT}_screening.csv")
print("\n===== summary (per-domain accuracy; rule 5)")
print(screen.to_string())

## 8 · Save, and push to GitHub when a token is attached

In [ ]:
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
out_dir = WORK / "results"

# every reported number gets a manifest row -- no row, no report (rule 10)
rows = ""
for model_id, df in frames_by_model.items():
    rows += (f"| {stamp} | {EXPERIMENT} | {model_id} | uniform {cache['fps']} FPS, short side "
             f"{RUN_SIDE[model_id] or cache['size']}, cap {MAX_FRAMES} | {df.n_frames.mean():.1f} avg | "
             f"FBBench n={len(df)} | {COMMIT} | {RESULTS[model_id].name} |\n")
(out_dir / "MANIFEST_ROWS.md").write_text(rows)
print(rows)

if PUSH_RESULTS and GH_TOKEN:
    dest = REPO / "results"
    for f in out_dir.glob(f"{EXPERIMENT}_*"):
        (dest / f.name).write_bytes(f.read_bytes())
    sh(["git", "pull", "--rebase", "origin", BRANCH], cwd=REPO, check=False)
    with open(dest / "MANIFEST.md", "a") as fh:
        fh.write(rows)
    # -f: results/* is gitignored apart from the manifest
    sh(["git", "add", "-f", "results/"], cwd=REPO)
    sh(["git", "commit", "-m", f"results: {EXPERIMENT} {len(frames_by_model)} runs"], cwd=REPO, check=False)
    sh(["git", "push", "origin", BRANCH], cwd=REPO)
    print("pushed to", BRANCH)
else:
    print("not pushed: results are in this notebook's output (/kaggle/working/results)")